In [1]:
import json
from pathlib import Path
import sys
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget

import spectra.data as data
from spectra.absorberspectrum import AbsorberSpectrum
from spectra.monomer import Monomer

# Initial design thoughts

Requirements

- Specify absorber(s) and photoinitiator(s) - these can all just go into a list because all that matters is their absorption spectrum; need a way to specify their concentration (molar or w/w)
- Specify monomer(s) and their concentrations (molar or w/w), or **their fractions (100%, 50%-50%, or 25%-30%-45%, for example)**
- Calculate absorption coefficient as a function of wavelength. Should be able to interpolate values so can sample at any wavelength or set of wavelengths

Possible ways of specifying materials and concentrations

- List of tuples with tuples of the form: (material object, concentration)
- **List of objects with objects having attributes of material, concentration_w/w_perc**, concentration_molar
- List of tuples with tuples of the form: (material object, mass in resin formulation)

Resin class provides:

- Absorption coefficient as a function of wavelength
- Molar concentrations of all materials?
- Weight fractions of all materials?

# Resin constituent material classes

In [2]:
@dataclass
class ResinConstituentAbsorber:
    """Container with 2 pieces of information:
    1. Absorber material
    2. Concentration w/w (percent) in resin
    
    Examples 
    --------
    1% Irgacure 819:
        `irgacure819 = ResinConstituentAbsorber(
            data.photoinitiators['irgacure819_in_PEGDA_2020-01-20'],
            1.0
        )`
    0.38% avobenzone:
        `avobenzone = ResinConstituentAbsorber(
            data.absorbers['avobenzone_in_PEGDA_2020-01-20'],
            0.38
        )`
    """
    material: AbsorberSpectrum
    concentration_ww_percent: (int, float)
    
    def __post_init__(self):
        
        # Enforce data types
        assert isinstance(self.material, AbsorberSpectrum)
        assert isinstance(self.concentration_ww_percent, (int, float))

@dataclass
class ResinConstituentMonomer:
    """Container with 2 pieces of information:
    1. Monomer material
    2. Concentration as percent of all MONOMERS in resin (not percent of total resin)
    
    Example: monomers for 60-40 PEGDA-HDDA resin:
        `pegda = ResinConstituentMonomer(data.monomers['PEGDA'], 60)`
        `hdda = ResinConstituentMonomer(data.monomers['HDDA'], 40)`
    """
    material: Monomer
    concentration_percent_monomer: (int, float)
    
    def __post_init__(self):
        
        # Enforce data types
        assert isinstance(self.material, Monomer)
        assert isinstance(self.concentration_percent_monomer, (int, float))


In [3]:
data.absorbers

{'avobenzone_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7fdc6190b590>,
 'avobenzone_in_PEGDA_2020-01-20': <spectra.absorberspectrum.AbsorberSpectrum at 0x7fdc614b6190>,
 'benetex_obplus_in PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7fdc502ad690>,
 'bls99-2_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7fdc401ea8d0>,
 'martius_yellow_in PEGDA_2020-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7fdc6190b610>,
 'nps_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7fdc6190b550>,
 'octocrylene_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7fdc6190bad0>,
 'phenazine_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7fdc6190b690>,
 'salicylaldehyde_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7fdc6190b650>,
 'sudanI_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7fdc6190b8d0>}

In [4]:
temp_absorber = ResinConstituentAbsorber(
    data.absorbers["avobenzone_in_PEGDA_2020-01-20"],
    0.38
)
temp_absorber

ResinConstituentAbsorber(material=<spectra.absorberspectrum.AbsorberSpectrum object at 0x7fdc614b6190>, concentration_ww_percent=0.38)

In [5]:
data.photoinitiators

{'irgacure819_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7fdc6190b490>,
 'irgacure819_in_PEGDA_2020-01-20': <spectra.absorberspectrum.AbsorberSpectrum at 0x7fdc6190b5d0>,
 'tmdpo_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7fdc6190b4d0>}

In [6]:
temp_photoinitiator = ResinConstituentAbsorber(
    data.absorbers["avobenzone_in_PEGDA_2020-01-20"],
    1.0
)
temp_photoinitiator

ResinConstituentAbsorber(material=<spectra.absorberspectrum.AbsorberSpectrum object at 0x7fdc614b6190>, concentration_ww_percent=1.0)

In [7]:
data.monomers

{'HDDA': Monomer(/Users/nordin/Documents/Projects/3D_printer_spectra_repos/3dprinter_spectra_and_tools/spectra/data/monomers/hdda.json),
 'PEGDA': Monomer(/Users/nordin/Documents/Projects/3D_printer_spectra_repos/3dprinter_spectra_and_tools/spectra/data/monomers/pegda.json),
 'TET': Monomer(/Users/nordin/Documents/Projects/3D_printer_spectra_repos/3dprinter_spectra_and_tools/spectra/data/monomers/tet.json)}

In [8]:
temp_monomer = ResinConstituentMonomer(
    data.monomers['PEGDA'],
    100
)
temp_monomer

ResinConstituentMonomer(material=Monomer(/Users/nordin/Documents/Projects/3D_printer_spectra_repos/3dprinter_spectra_and_tools/spectra/data/monomers/pegda.json), concentration_percent_monomer=100)

In [9]:
vars(temp_monomer.material)

{'_json_file': PosixPath('/Users/nordin/Documents/Projects/3D_printer_spectra_repos/3dprinter_spectra_and_tools/spectra/data/monomers/pegda.json'),
 '_from_json': {'Name': 'PEGDA',
  'Long name': 'poly ethylene glycol diacrylate',
  'Density g/L': 1110},
 'name': 'PEGDA',
 'long_name': 'poly ethylene glycol diacrylate',
 'density': 1110,
 'density_units': 'g/L'}

In [10]:
temp_monomer.material.density

1110

In [11]:
temp_monomer.material.density_units

'g/L'

# Resin class

In [ ]:
def Resin:
    """Combine materials into a resin and calculate its absorption coefficient
    as a function of wavelength.
    
    Parameters
    ----------
    monomers - ResinConstituentMonomer of list of ResinConstituentMonomer
    photoinitiators - ResinConstituentAbsorber or list of ResinConstituentAbsorber
    absorbers - ResinConstituentAbsorber or list of ResinConstituentAbsorber
    """
    def __init__(self):
        
        # Inputs: monomer(s), photoinitiator(s), absorbers(s)
        
        
        # For absorber, photoinitiator, monomer, determine if it's one or more by looking at type - 
        # add absorbers and photoinitiators to a list of absorbers
        
        # Calculate molar concentration for absorbers, photoinitiators in list of absorbers
        
        # Calculate absorption coefficient as a function of wavelength
        
        # Make absorption coefficient so can linearly interpolate values
        